# 22. Opening-night rosters, 2026-27

Data step before the season simulator. The simulator starts on **2026-27 opening night** in a **"no disputes, full health"** world: every player is on the team that holds him, even if he's suspended, holding out or an unsigned RFA. Ryan's main experiment is Detroit with vs. without Dylan Larkin (trade request, suspended) and Simon Edvinsson (unsigned RFA in Sweden).

**Why this is needed.** Until now, NHL players carried their last 2025-26 team (summer trades and signings missing) and prospects their draft team (rights trades and expired rights missing).

**Sources** (NHL API only, downloaded once, polite and resumable, saved in `data/current_raw/`, gitignored):

- each team's **active roster today** (`/v1/roster/{TEAM}/current`; it leaves out injured, suspended and AHL players),
- a **fresh profile** for every player we project plus everyone on a roster. The profile's `currentTeamAbbrev` is the NHL's own record of who holds a player, **including unsigned draft rights and traded rights**, and its 2026-27 rows show where he's playing,
- the **2026-27 schedule** (opening night),
- **game logs** for the few players who played NHL games for more than one team this season (to date the move).

There's no NHL endpoint for transactions or for opening-night rosters, so in-season moves come from game logs and a hand-kept trade log.

**Three hand-kept files** (gitignored, created here if missing, never overwritten):

- `data/trades_2026-27.csv`: in-season trades, one row per player moved. The baseline roster **undoes** them (opening night); the simulator re-applies them.
- `data/roster_overrides.csv`: hand corrections (playerId, name, team, status, note); overrides win.
- `data/affiliates_2026-27.csv`: written by this notebook from the data (AHL/ECHL club -> NHL parent).

## 1. Setup

In [1]:
import sys
sys.path.append("../src")

import json
import numpy as np
import pandas as pd
from pathlib import Path

from forecast import history_table, current_projections, season_age, GOALS_PER_WIN
from edge_data import (download_rosters, download_current_profiles, download_game_logs, download_schedule,
                       build_current_tables, nhl_team_names, affiliate_table, assign_teams, first_nhl_teams,
                       build_bio_table, MINOR_LEAGUES)
from nhle import load_profiles
from development import (nb19_models, project_skater_prospects, nb21_goalie_model, goalie_prior_war_60,
                         load_goalie_seasons, first_nhl_goalie_season, goalie_features, pre_nhl_save_pct, NB21_K)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

DATA = Path("../data")
OUT = DATA                                   # where the roster, affiliate and new-player files are saved
RAW = DATA / "current_raw"                   # the snapshot (rosters, profiles, game logs, schedule)
SEASON = "20262027"

proj = current_projections(DATA, include_new=False)       # everyone notebooks 18, 19 and 21 project
print(f"Players we project: {len(proj):,} ({(~proj['prospect']).sum():,} NHL, {proj['prospect'].sum():,} prospects)")

Players we project: 2,495 (1,328 NHL, 1,167 prospects)


## 2. Downloads

Everything is saved one file per team or player, stamped with the download time. A rerun reads the saved files and only fetches what's missing, so this snapshot stays fixed. (To refresh it, move `data/current_raw/` aside and rerun.) The first run takes about 40 minutes for the profiles.

In [2]:
TEAMS = sorted(set(proj.loc[~proj["prospect"], "team"]) - {"ARI"})      # the 32 current teams (Arizona is now Utah)
download_rosters(TEAMS, RAW / "rosters")
sched = download_schedule("2026-10-06", RAW / "schedule.json")
opening_week = download_schedule(sched["regularSeasonStartDate"], RAW / "schedule_opening_week.json")

roster_ids = set()
for f in (RAW / "rosters").glob("*.json"):
    d = json.loads(f.read_text())
    roster_ids |= {p["id"] for k in ["forwards", "defensemen", "goalies"] for p in d[k]}
overrides_path, trades_path = DATA / "roster_overrides.csv", DATA / "trades_2026-27.csv"
hand_ids = set()
for path in [overrides_path, trades_path]:
    if path.exists():
        hand_ids |= set(pd.read_csv(path)["playerId"])
ids = sorted(set(proj["playerId"]) | roster_ids | hand_ids)
print(f"Profiles needed: {len(ids):,} (projected {len(proj):,}, on a roster today {len(roster_ids):,}, "
      f"on a roster but not projected {len(roster_ids - set(proj['playerId']))})")
failed = download_current_profiles(ids, RAW / "players")

Rosters saved: 32
Profiles needed: 2,503 (projected 2,495, on a roster today 797, on a roster but not projected 8)
2503 already saved, 0 to download (about 0 minutes)
Done, no failures.


## 3. Opening night

In [3]:
OPENING_NIGHT = sched["regularSeasonStartDate"]
first_day = [w for w in opening_week["gameWeek"] if w["date"] == OPENING_NIGHT][0]
print(f"Opening night: {OPENING_NIGHT}")
for g in first_day["games"]:
    print(f"  {g['awayTeam']['abbrev']} at {g['homeTeam']['abbrev']}" + ("" if g["gameType"] == 2 else f" (game type {g['gameType']})"))

players, rows, rosters = build_current_tables(RAW / "players", RAW / "rosters")
print(f"\nProfiles read: {len(players):,}; 2026-27 season rows: {len(rows):,}; on active rosters: {len(rosters):,} "
      f"(per team {rosters['roster_team'].value_counts().min()} to {rosters['roster_team'].value_counts().max()})")
print(f"Rosters downloaded {rosters['roster_as_of'].min()}; profiles {players['profile_as_of'].min()} to {players['profile_as_of'].max()}")

Opening night: 2026-09-29
  FLA at CAR
  MTL at TOR
  NYR at BOS
  VAN at EDM
  CHI at VGK

Profiles read: 2,503; 2026-27 season rows: 2,454; on active rosters: 797 (per team 22 to 29)
Rosters downloaded 2026-11-06 11:48; profiles 2026-11-06 11:49 to 2026-11-06 12:14


## 4. AHL and ECHL affiliates, from the data

There's no NHL endpoint for affiliations, so they're read off the profiles: for each 2026-27 AHL or ECHL club, the parent is the **most common NHL team on its players' profiles** (players on NHL contracts show their NHL team). A club needs 3+ such players and a 60% majority; otherwise its parent is unknown. ECHL clubs often serve two NHL teams, so some will be unknown.

In [4]:
affiliates = affiliate_table(players, rows)
affiliates.to_csv(OUT / "affiliates_2026-27.csv", index=False)
for league in ["AHL", "ECHL"]:
    a = affiliates[affiliates["league"] == league]
    print(f"{league}: {len(a)} clubs, parent found for {a['parent'].notna().sum()}")
print()
print(affiliates.assign(share=affiliates["share"].round(2)).to_string(index=False))
ahl = affiliates[(affiliates["league"] == "AHL") & affiliates["parent"].notna()]
dupes = ahl["parent"].value_counts()
print("\nNHL teams with more than one AHL club:", dupes[dupes > 1].to_dict() or "none",
      "| NHL teams with none:", sorted(set(TEAMS) - set(ahl["parent"])) or "none")

AHL: 32 clubs, parent found for 32
ECHL: 27 clubs, parent found for 6

league                           club  players  nhl_signed top_parent  share parent other_parents
   AHL             Abbotsford Canucks       16          13        VAN   1.00    VAN              
   AHL            Bakersfield Condors       11          10        EDM   1.00    EDM              
   AHL            Belleville Senators       20          19        OTT   1.00    OTT              
   AHL              Calgary Wranglers       18          18        CGY   1.00    CGY              
   AHL             Charlotte Checkers       21          20        FLA   1.00    FLA              
   AHL                 Chicago Wolves       22          20        CAR   1.00    CAR              
   AHL             Cleveland Monsters       18          18        CBJ   1.00    CBJ              
   AHL     Coachella Valley Firebirds       20          18        SEA   1.00    SEA              
   AHL                Colorado Eagles       16 

## 5. The trade log and the overrides

Both files are hand-kept and created here **only if missing** (a rerun never overwrites your edits).

- **Trade log:** the Hellebuyck trade as reported on October 23, 2026. Picks are skipped. Alexander Command is a prospect (the 12th overall pick in 2026), so the trade moves his rights.
- **Overrides:** only cases with evidence. Larkin and Edvinsson are Red Wings in a "no disputes" world; the API shows both on Detroit's books but off the active roster, and it can't tell suspended or unsigned players from injured ones, so their statuses come from Ryan.

In [5]:
if not trades_path.exists():
    pd.DataFrame([
        {"trade_id": "T1", "date": "2026-10-23", "playerId": 8476945, "name": "Connor Hellebuyck", "from_team": "WPG", "to_team": "NJD", "note": "picks skipped"},
        {"trade_id": "T1", "date": "2026-10-23", "playerId": 8482110, "name": "Dawson Mercer", "from_team": "NJD", "to_team": "WPG", "note": ""},
        {"trade_id": "T1", "date": "2026-10-23", "playerId": 8482076, "name": "Nico Daws", "from_team": "NJD", "to_team": "WPG", "note": ""},
        {"trade_id": "T1", "date": "2026-10-23", "playerId": 8475760, "name": "Nick Bjugstad", "from_team": "NJD", "to_team": "WPG", "note": ""},
        {"trade_id": "T1", "date": "2026-10-23", "playerId": 8486030, "name": "Alexander Command", "from_team": "NJD", "to_team": "WPG",
         "note": "prospect, 2026 12th overall pick"},
    ]).to_csv(trades_path, index=False)
    print("Created", trades_path.name)
if not overrides_path.exists():
    pd.DataFrame([
        {"playerId": 8477946, "name": "Dylan Larkin", "team": "DET", "status": "suspended / holdout",
         "note": "Ryan: trade request, suspended; profile DET, 10 NHL games, off the active roster"},
        {"playerId": 8482762, "name": "Simon Edvinsson", "team": "DET", "status": "unsigned RFA",
         "note": "Ryan: unsigned RFA in Sweden; profile DET, no 2026-27 games, off the active roster"},
    ]).to_csv(overrides_path, index=False)
    print("Created", overrides_path.name)
trades = pd.read_csv(trades_path)
overrides = pd.read_csv(overrides_path)
print("\nTrade log:")
print(trades.to_string(index=False))
print("\nOverrides:")
print(overrides.to_string(index=False))

Created trades_2026-27.csv
Created roster_overrides.csv

Trade log:
trade_id       date  playerId              name from_team to_team                             note
      T1 2026-10-23   8476945 Connor Hellebuyck       WPG     NJD                    picks skipped
      T1 2026-10-23   8482110     Dawson Mercer       NJD     WPG                              NaN
      T1 2026-10-23   8482076         Nico Daws       NJD     WPG                              NaN
      T1 2026-10-23   8475760     Nick Bjugstad       NJD     WPG                              NaN
      T1 2026-10-23   8486030 Alexander Command       NJD     WPG prospect, 2026 12th overall pick

Overrides:
 playerId            name team              status                                                                               note
  8477946    Dylan Larkin  DET suspended / holdout   Ryan: trade request, suspended; profile DET, 10 NHL games, off the active roster
  8482762 Simon Edvinsson  DET        unsigned RFA Ryan: u

## 6. The team rules

**Team today**, in order: (1) on an active roster today; (2) the team on his NHL profile; (3) his 2026-27 AHL/ECHL club's NHL parent; (4) for a draftee nobody lists, the CBA check in section 7; else none.

**Opening night**, starting from the team today:

- **In-season moves with games:** a player whose first 2026-27 NHL game was for another team starts there (from his game log). These are listed as **candidate trade-log rows** for Ryan to confirm; they're not added to the log automatically.
- **Logged trades:** a move in the trade log dated after opening night is undone (Hellebuyck starts as a Jet).
- **Overrides** win.

**Status:**

| Status | Meaning |
|---|---|
| NHL roster | on an active roster today |
| NHL org, off active roster | his profile lists an NHL team and he has NHL games, but he isn't on the active roster or in the AHL: injured, suspended, holding out or unsigned (the API can't tell which) |
| AHL/ECHL (NHL contract) | playing in the AHL/ECHL, with an NHL team on his profile |
| AHL/ECHL (AHL contract) | playing in the AHL/ECHL, no NHL team on his profile; his team is the club's NHL parent |
| rights held, playing elsewhere | his profile lists an NHL team and he plays junior, college or in Europe: usually unsigned draft rights, sometimes a signed player loaned back |
| rights held, profile blank (CBA) | a draftee whose profile lists no team, but the CBA rule says his draft team still holds his rights (rule 4); uncertain, check these |
| rights held, no 2026-27 games yet | an NHL team holds him and he hasn't played this season |
| rights likely expired | a draftee nobody lists, and the CBA says his rights have run out |
| not with an NHL organization | none of the above: retired, or playing somewhere with no NHL team. Kept in the table, out of simulator rosters |

Overrides can set any status (Larkin "suspended / holdout", Edvinsson "unsigned RFA").

In [6]:
team_names = nhl_team_names(players)
nhl_rows = rows[rows["league"] == "NHL"].assign(team=lambda d: d["club"].map(team_names))
unmapped = nhl_rows[nhl_rows["team"].isna()]["club"].unique()
assert len(unmapped) == 0, f"NHL club names with no abbreviation: {unmapped}"

# First pass without in-season moves, to find who needs a game log
base, _ = assign_teams(players, rows, rosters, affiliates, team_names, trades.iloc[:0], overrides.iloc[:0], OPENING_NIGHT, pd.Series(dtype=object))
teams26 = nhl_rows.groupby("playerId")["team"].agg(set)
today = base.set_index("playerId")["team_today"]
need_log = [pid for pid, t in teams26.items() if len(t) > 1 or today.get(pid) not in t]
print(f"Players with 2026-27 NHL games for more than one team, or for a team other than today's: {len(need_log)}")
download_game_logs(need_log, SEASON, RAW / "game_logs")
first_team, moves = first_nhl_teams(RAW / "game_logs")
first_team = first_team.reindex(need_log).dropna()

ros, trade_check = assign_teams(players, rows, rosters, affiliates, team_names, trades, overrides, OPENING_NIGHT, first_team)
print("\nRule that decided each player's opening-night team:")
print(ros["rule"].str.replace(r"override: .*", "override", regex=True).str.replace(r"trade log .*", "trade log undone", regex=True)
      .value_counts().to_string())
print("\nStatus:")
print(ros["status"].value_counts().to_string())

Players with 2026-27 NHL games for more than one team, or for a team other than today's: 10
10 already saved, 0 to download (about 0 minutes)
Done, no failures.

Rule that decided each player's opening-night team:
rule
team on his NHL profile                              1228
active roster today                                   787
no 2026-27 games, no NHL team                         159
no NHL team, playing elsewhere                        130
CBA rights expired (profile blank)                     91
AHL/ECHL club's NHL parent                             56
draft team, CBA rights still held (profile blank)      37
first 2026-27 NHL team (in-season move)                 8
trade log undone                                        5
override                                                2

Status:
status
NHL roster                           797
rights held, playing elsewhere       643
AHL/ECHL (NHL contract)              526
not with an NHL organization         289
rights likely expired

In [7]:
print("Trade-log check (today's team should be each player's to_team):")
print(trade_check.to_string(index=False))
bad = trade_check[~trade_check["matches"]]
print("Mismatches:", "none" if bad.empty else bad["name"].tolist())

logged = set(trades["playerId"])
cand = moves[moves["first_game"] > OPENING_NIGHT].merge(players[["playerId", "name"]], on="playerId")
cand = cand[~cand["playerId"].isin(logged)]
print(f"\nCandidate trade-log rows (in-season moves seen in game logs, not in the log): {len(cand)}")
print(cand[["name", "playerId", "from_team", "to_team", "first_game"]].sort_values("first_game").to_string(index=False))

diff = ros[ros["roster_team"].notna() & ros["profile_team"].notna() & (ros["roster_team"] != ros["profile_team"])]
print(f"\nOn a roster for a different team than his profile says: {len(diff)}",
      diff[["name", "roster_team", "profile_team"]].to_string(index=False) if len(diff) else "")

Trade-log check (today's team should be each player's to_team):
trade_id              name from_team to_team team_today  matches
      T1 Connor Hellebuyck       WPG     NJD        NJD     True
      T1     Dawson Mercer       NJD     WPG        WPG     True
      T1         Nico Daws       NJD     WPG        WPG     True
      T1     Nick Bjugstad       NJD     WPG        WPG     True
      T1 Alexander Command       NJD     WPG        WPG     True
Mismatches: none

Candidate trade-log rows (in-season moves seen in game logs, not in the log): 6
           name  playerId from_team to_team first_game
Filip Hallander   8480842       PIT     EDM 2026-10-10
 Jayden Struble   8481593       MTL     NSH 2026-10-10
    Daemon Hunt   8482094       MIN     NSH 2026-10-25
 Jack St. Ivany   8481030       WPG     WSH 2026-10-28
   Shane Wright   8483524       SEA     NSH 2026-11-03
   Matthew Wood   8484241       NSH     SEA 2026-11-04

On a roster for a different team than his profile says: 0 


## 7. The CBA check for unsigned draft rights

The profile is the evidence for who holds a draftee. The CBA is used as a check, and to decide players whose profile lists no team. **The rules below are from memory** of CBA Article 8.6 (2012 CBA, extended by the 2020 and 2025 MOUs); I couldn't check the text (this notebook only calls NHL endpoints), so they're approximate:

- **College (NCAA):** the team holds his rights while he's in college, up to August 15 after his class graduates. *Unsure:* what happens if he leaves school early, and how the 2025 change letting CHL players into the NCAA interacts with this.
- **Europe** (including the KHL and European junior leagues): four years, until June 1 of the draft year + 4. *Unsure* for players drafted at 20 or older.
- **Everyone else** (CHL, USHL, other junior): two years, until June 1 of the draft year + 2.

In [8]:
dr = ros[ros["draft_year"].notna() & ~ros["status"].isin(["NHL roster", "NHL org, off active roster"])
         & ~ros["league_now"].isin(MINOR_LEAGUES | {"NHL"}) & (ros["nhl_gp_career"] == 0)]
flag1 = dr[dr["profile_team"].notna() & ~dr["cba_held"] & dr["league_now"].notna()]
flag2 = dr[dr["profile_team"].isna() & dr["cba_held"] & dr["league_now"].notna()]
cols = ["name", "draft_year", "draft_team", "profile_team", "league_now", "club_now", "cba_rule", "status"]
print(f"Unsigned-looking draftees playing outside the NHL/AHL: {len(dr):,}")
print(f"\nProfile lists a team, but the CBA says the rights ran out ({len(flag1)}); kept with the profile's team (maybe signed and loaned, or the rule is off):")
print(flag1[cols].sort_values("draft_year").head(25).to_string(index=False))
print(f"\nProfile lists no team, but the CBA says the rights are still held ({len(flag2)}); given the draft team (Arizona's picks to Utah):")
print(flag2[cols].sort_values("draft_year").to_string(index=False))

Unsigned-looking draftees playing outside the NHL/AHL: 818

Profile lists a team, but the CBA says the rights ran out (29); kept with the profile's team (maybe signed and loaned, or the rule is off):
               name  draft_year draft_team profile_team league_now               club_now                   cba_rule                         status
       Topi Niemela      2020.0        TOR          TOR        SHL          IF Björklöven Europe: until June 1, 2024 rights held, playing elsewhere
         Eemil Viro      2020.0        DET          DET        SHL         Malmö Redhawks Europe: until June 1, 2024 rights held, playing elsewhere
        Calle Clang      2020.0        PIT          ANA        SHL               Rögle BK Europe: until June 1, 2024 rights held, playing elsewhere
           Leo Loof      2020.0        STL          STL      Liiga                  Ilves Europe: until June 1, 2024 rights held, playing elsewhere
   Nikita Guslistov      2021.0        CAR          CAR     

## 8. New players

Anyone on a 2026-27 active roster (or in the overrides or trade log) whom we don't project is **new to our data**. They're projected the way notebook 19 projects prospects (skaters) and notebook 21 projects goalie prospects, with the same code moved into `src/development.py`. First, two checks that the moved code is the same: it must reproduce notebook 19's 1,167 prospects and notebook 21's 139 goalie prospects exactly.

In [9]:
history = history_table(pd.read_csv(DATA / "war_by_season_2008_2025.csv"), pd.read_csv(DATA / "goalie_war_2008_2025.csv"))
bios = build_bio_table(DATA / "bio_raw", 2026)
profiles = load_profiles(DATA / "bio_raw")
careers = pd.read_csv(DATA / "career_totals.csv")
nhle_factors = pd.read_csv(DATA / "nhle_factors.csv")
nhle_mult = pd.read_csv(DATA / "nhle_age_multipliers.csv")
dev_ranges = pd.read_csv(DATA / "dev_ranges.csv")
luck = dev_ranges[dev_ranges["kind"] == "luck_fit"].set_index("group")[["a", "b"]]
rookie_range = dev_ranges[dev_ranges["kind"] == "rookie"]
pros = pd.read_csv(DATA / "prospects_2026-27.csv")
gproj = pd.read_csv(DATA / "goalie_projections_2026-27.csv")

# Check 1: notebook 19's skater prospects
sk_models = nb19_models(history, bios, profiles, careers, nhle_factors, nhle_mult)
all_profiles = pd.concat([profiles, load_profiles(DATA / "prospect_raw")], ignore_index=True)
bios_all = pd.concat([bios, build_bio_table(DATA / "prospect_raw", 2026)]).drop_duplicates("playerId")
sk = pros[pros["group"] != "G"]
re19, fill25 = project_skater_prospects(sk[["playerId", "group"]], history, bios_all, all_profiles, nhle_factors, nhle_mult,
                                        sk_models, rookie_range, luck)
m = sk.set_index("playerId").join(re19.set_index("playerId"), rsuffix="_re")
diff19 = max(np.nanmax(np.abs(m[c] - m[c + "_re"])) for c in ["proj_war_60", "d_rank_prior", "war_60_p10", "war_60_p90"])
assert diff19 < 1e-12 and (m["proj_war_60"].isna() == m["proj_war_60_re"].isna()).all(), diff19
print(f"Notebook 19 reproduced: {len(sk):,} skater prospects, largest difference {diff19:.1e} WAR/60")

# Check 2: notebook 21's goalie prospects
gseasons = load_goalie_seasons(DATA / "bio_raw", DATA / "prospect_raw")
g_model = nb21_goalie_model(history, bios, profiles, gseasons)
pre_sv = pre_nhl_save_pct(gseasons, g_model["first_nhl"], before=2026, shrink_shots=NB21_K)
gp = pros[pros["group"] == "G"][["playerId", "age"]].merge(pre_sv[["playerId", "pre_sv"]], on="playerId", how="left").assign(experience=0.0)
gp["re"] = goalie_prior_war_60(gp, g_model).to_numpy()
m = gp.set_index("playerId").join(gproj.set_index("playerId")["new_proj_war_60"])
diff21 = np.abs(m["re"] - m["new_proj_war_60"]).max()
assert diff21 < 1e-12, diff21
print(f"Notebook 21 reproduced: {len(gp)} goalie prospects, largest difference {diff21:.1e} WAR/60 "
      f"(debut age {g_model['debut_age']:.0f}, age floor {g_model['age_floor']})")

Notebook 19 reproduced: 1,028 skater prospects, largest difference 9.9e-17 WAR/60
Notebook 21 reproduced: 139 goalie prospects, largest difference 9.7e-17 WAR/60 (debut age 24, age floor 20)


In [10]:
projected = set(proj["playerId"])
want = (ros["status"].isin(["NHL roster"]) | ros["playerId"].isin(hand_ids)) & ~ros["playerId"].isin(projected)
new = ros[want][["playerId", "name", "position", "birth_date", "team_opening_night", "draft_year", "draft_team", "draft_pick", "nhl_gp_career"]].copy()
new["group"] = np.where(new["position"] == "G", "G", np.where(new["position"] == "D", "D", "F"))
new["age"] = season_age(new["birth_date"], np.full(len(new), 2026))
print(f"New to our data: {len(new)} ({', '.join(f'{g} {n}' for g, n in new['group'].value_counts().items())})")

# Their fresh profiles (only finished seasons count: 2025-26 and earlier)
new_profiles = load_profiles(RAW / "players")
new_profiles = new_profiles[new_profiles["playerId"].isin(new["playerId"])]
new_bios = build_bio_table(RAW / "players", 2026)
new_bios = new_bios[new_bios["playerId"].isin(new["playerId"])]

# Skaters: notebook 19's pipeline, with notebook 19's NHLe fill (25th percentile of its prospects)
nsk = new[new["group"] != "G"]
sk_out, _ = project_skater_prospects(nsk[["playerId", "group"]], history, pd.concat([bios_all, new_bios]).drop_duplicates("playerId"),
                                     pd.concat([all_profiles, new_profiles], ignore_index=True), nhle_factors, nhle_mult,
                                     sk_models, rookie_range, luck, fill25=fill25)

# Goalies: notebook 21's prior (experience, age, shrunk pre-NHL save%), debut-age rule for no NHL experience
ng = new[new["group"] == "G"]
g_out = pd.DataFrame(columns=["playerId"])
if len(ng):
    gs_new = load_goalie_seasons(RAW / "players")
    gs_all = pd.concat([gseasons, gs_new[gs_new["playerId"].isin(ng["playerId"])]], ignore_index=True)
    fn_all = first_nhl_goalie_season(history, gs_all)
    gf = goalie_features(ng[["playerId"]].assign(season=2026), history, pd.concat([bios, new_bios]).drop_duplicates("playerId"),
                         pd.concat([profiles, new_profiles], ignore_index=True), gs_all, fn_all, shrink_shots=NB21_K)
    gf["proj_war_60"] = goalie_prior_war_60(gf, g_model).to_numpy()
    g_out = gf[["playerId", "experience", "pre_sv", "proj_war_60"]].assign(prior_type="goalie prior (nb21)")

# 2025-26 league, games and points (his main league that season), for the record
p25 = new_profiles[(new_profiles["season"] == 2025) & (new_profiles["gameTypeId"] == 2)]
main = p25.groupby(["playerId", "league"], as_index=False)[["gp", "points"]].sum().sort_values("gp", ascending=False).drop_duplicates("playerId")
newp = new.merge(pd.concat([sk_out, g_out], ignore_index=True), on=["playerId"], how="left", suffixes=("", "_x"))
newp = newp.merge(main.rename(columns={"league": "league_2025_26", "gp": "gp_2025_26", "points": "points_2025_26"}), on="playerId", how="left")
newp = newp.rename(columns={"nhle_ppg": "nhle_last2", "draft_pick": "pick", "team_opening_night": "team"})
NEW_COLS = ["playerId", "name", "position", "group", "age", "team", "draft_year", "pick", "draft_team", "nhl_gp_career", "experience",
            "league_2025_26", "gp_2025_26", "points_2025_26", "nhle_last2", "nhle_missing", "prior_type", "proj_war_60",
            "war_60_p10", "war_60_p50", "war_60_p90", "d_rank_prior"]
newp = newp[NEW_COLS]
assert newp["proj_war_60"].notna().all()
newp.to_csv(OUT / "new_players_2026-27.csv", index=False)
print(newp.drop(columns=["war_60_p50"]).round(3).to_string(index=False))

New to our data: 8 (F 6, G 2)
 playerId           name position group  age team  draft_year  pick draft_team  nhl_gp_career  experience league_2025_26  gp_2025_26  points_2025_26  nhle_last2 nhle_missing          prior_type  proj_war_60  war_60_p10  war_60_p90  d_rank_prior
  8480029 Alex Formenton        L     F 27.0  EDM      2017.0  47.0        OTT            126           2             NL        42.0            23.0       0.391        False      (b), rule (iv)        0.052      -0.016       0.131           NaN
  8481538 Judd Caulfield        R     F 25.0  ANA      2019.0 145.0        PIT             14           0            AHL        71.0            38.0       0.184        False      (b), rule (iv)        0.012      -0.056       0.091           NaN
  8482465   Rémi Poirier        G     G 25.0  WSH      2020.0 185.0        DAL              0           0            AHL        52.0             0.0         NaN          NaN goalie prior (nb21)        0.040         NaN         NaN     

## 9. Save the roster table

`data/rosters_2026-27.csv` (gitignored): one row per player with his last 2025-26 team (from our projections), team today, **team on opening night**, status, the rule that decided it, and the as-of dates.

In [11]:
last_team = proj.set_index("playerId")["team"]               # NHL players: last 2025-26 team; prospects: draft team
ros["team_last_2025_26"] = ros["playerId"].map(last_team.where(~proj.set_index("playerId")["prospect"]))
ros["opening_night"] = OPENING_NIGHT
ROS_COLS = ["playerId", "name", "position", "team_last_2025_26", "team_today", "team_opening_night", "status", "rule",
            "league_now", "club_now", "roster_as_of", "profile_as_of", "opening_night"]
out = ros[ROS_COLS].sort_values(["team_opening_night", "status", "name"])
assert out["playerId"].is_unique
out.to_csv(OUT / "rosters_2026-27.csv", index=False)
print(f"Saved rosters_2026-27.csv: {len(out):,} players; with an NHL team on opening night: {out['team_opening_night'].notna().sum():,}")

Saved rosters_2026-27.csv: 2,503 players; with an NHL team on opening night: 2,123


## 10. Checks

In [12]:
NHL_LEVEL = ["NHL roster", "NHL org, off active roster", "suspended / holdout", "unsigned RFA"]
o = ros[ros["team_opening_night"].isin(TEAMS)]
counts = pd.crosstab(o["team_opening_night"], o["status"])
counts["NHL level"] = counts[[c for c in NHL_LEVEL if c in counts]].sum(axis=1)
counts["organization"] = counts.drop(columns="NHL level").sum(axis=1)
print("Opening-night counts by team and status (NHL level = roster + off the active roster + disputes):")
print(counts.to_string())
print(f"\nNHL roster per team: {counts['NHL roster'].min()} to {counts['NHL roster'].max()}; "
      f"NHL level: {counts['NHL level'].min()} to {counts['NHL level'].max()}")

have = projected | set(newp["playerId"])
nhl_open = ros[ros["status"].isin(NHL_LEVEL) & ros["team_opening_night"].notna()]
missing = nhl_open[~nhl_open["playerId"].isin(have)]
assert missing.empty, missing[["name", "team_opening_night", "status"]]
print(f"Every NHL-level player on opening night has a projection ({len(nhl_open):,} players).")

Opening-night counts by team and status (NHL level = roster + off the active roster + disputes):
status              AHL/ECHL (AHL contract)  AHL/ECHL (NHL contract)  NHL org, off active roster  NHL roster  rights held, no 2026-27 games yet  rights held, playing elsewhere  rights held, profile blank (CBA)  suspended / holdout  unsigned RFA  NHL level  organization
team_opening_night                                                                                                                                                                                                                                                           
ANA                                       2                       14                           2          27                                  1                              23                                 4                    0             0         29            73
BOS                                       1                       15                         

In [13]:
SHOW = ["name", "position", "status", "team_today", "league_now", "club_now", "rule"]
det = ros[ros["team_opening_night"] == "DET"].sort_values("name")
parts = {"NHL roster (disputes included)": det["status"].isin(["NHL roster", "suspended / holdout", "unsigned RFA"]),
         "Signed, not on the active roster": det["status"] == "NHL org, off active roster",
         "AHL/ECHL": det["status"].str.startswith("AHL/ECHL"),
         "Rights held (junior, college, Europe or no games yet)": det["status"].str.startswith("rights held")}
for title, sel in parts.items():
    print(f"\nDetroit, {title} ({sel.sum()}):")
    print(det[sel][SHOW].to_string(index=False))
other = det[~pd.concat(parts.values(), axis=1).any(axis=1)]
if len(other):
    print(f"\nDetroit, other ({len(other)}):")
    print(other[SHOW].to_string(index=False))


Detroit, NHL roster (disputes included) (28):
                    name position              status team_today league_now              club_now                                                                                         rule
        Albert Johansson        D          NHL roster        DET        NHL     Detroit Red Wings                                                                          active roster today
          Alex DeBrincat        R          NHL roster        DET        NHL     Detroit Red Wings                                                                          active roster today
         Andreas Englund        D          NHL roster        DET        NHL     Detroit Red Wings                                                                          active roster today
             Andrew Copp        C          NHL roster        DET        NHL     Detroit Red Wings                                                                          active roster toda

In [14]:
for team in ["WPG", "NJD"]:
    t = ros[(ros["team_opening_night"] == team) & ros["status"].isin(NHL_LEVEL)].sort_values(["position", "name"])
    print(f"\n{team} on opening night, NHL level ({len(t)}):")
    print(t[["name", "position", "status", "team_today", "rule"]].to_string(index=False))
c = ros[ros["name"] == "Alexander Command"]
print("\nAlexander Command:", c[["team_today", "team_opening_night", "status", "league_now", "rule"]].to_dict("records"))


WPG on opening night, NHL level (26):
                 name position                     status team_today                                    rule
           Adam Lowry        C                 NHL roster        WPG                     active roster today
         Brad Lambert        C NHL org, off active roster        WPG                 team on his NHL profile
        Brayden Yager        C                 NHL roster        WPG                     active roster today
        Cole Perfetti        C                 NHL roster        WPG                     active roster today
      Gabriel Vilardi        C                 NHL roster        WPG                     active roster today
       Mark Scheifele        C                 NHL roster        WPG                     active roster today
        Morgan Barron        C                 NHL roster        WPG                     active roster today
         Viggo Björck        C                 NHL roster        WPG                     

In [15]:
s = ros[ros["name"] == "Redmond Savage"]
print("Redmond Savage:")
print(s[["playerId", "draft_team", "draft_year", "profile_team", "is_active", "league_now", "club_now", "team_today",
         "team_opening_night", "status", "rule"]].T.to_string())

Redmond Savage:
                                          1109
playerId                               8482662
draft_team                                 DET
draft_year                              2021.0
profile_team                               NaN
is_active                                False
league_now                                 AHL
club_now                   Rochester Americans
team_today                                 BUF
team_opening_night                         BUF
status                 AHL/ECHL (AHL contract)
rule                AHL/ECHL club's NHL parent


In [16]:
moved = ros[ros["team_last_2025_26"].notna() & ros["team_opening_night"].notna()
            & (ros["team_last_2025_26"] != ros["team_opening_night"])].merge(proj[["playerId", "proj_war_60"]], on="playerId")
print(f"NHL players whose team changed since 2025-26: {len(moved)}. The 15 highest projected:")
print(moved.nlargest(15, "proj_war_60")[["name", "position", "team_last_2025_26", "team_opening_night", "status", "rule"]]
      .to_string(index=False))

NHL players whose team changed since 2025-26: 279. The 15 highest projected:
              name position team_last_2025_26 team_opening_night                  status                    rule
      Jordan Kyrou        R               STL                WSH              NHL roster     active roster today
   Pavel Dorofeyev        R               VGK                NYR              NHL roster     active roster today
     Brady Tkachuk        L               OTT                FLA              NHL roster     active roster today
    Anthony Mantha        R               PIT                NJD              NHL roster     active roster today
  Kirill Marchenko        R               CBJ                TOR              NHL roster     active roster today
    Nick Robertson        L               TOR                PIT              NHL roster     active roster today
        JJ Peterka        R               UTA                BOS              NHL roster     active roster today
Oliver Bjorkstrand 

In [17]:
gone = ros[ros["status"] == "not with an NHL organization"].merge(proj[["playerId", "prospect", "proj_war_60"]], on="playerId")
print(f"Projected players now not with an NHL organization: {len(gone)} "
      f"({(~gone['prospect']).sum()} NHL players, {gone['prospect'].sum()} prospects)")
print("\nNHL players (highest projection first), for Ryan to confirm:")
print(gone[~gone["prospect"]].sort_values("proj_war_60", ascending=False)
      [["name", "position", "team_last_2025_26", "is_active", "league_now", "club_now", "rule"]].head(40).to_string(index=False))
for name in ["Patrick Kane", "David Perron", "James van Riemsdyk", "Cam Talbot"]:
    r = ros[ros["name"] == name]
    if len(r):
        print(f"{name}: {r['team_opening_night'].iloc[0]}, {r['status'].iloc[0]} ({r['rule'].iloc[0]})")

Projected players now not with an NHL organization: 289 (246 NHL players, 43 prospects)

NHL players (highest projection first), for Ryan to confirm:
              name position team_last_2025_26  is_active league_now                   club_now                           rule
      Sonny Milano        L               WSH      False         NL                    SC Bern no NHL team, playing elsewhere
     Daniel Sprong        R               NJD      False        KHL Avtomobilist Yekaterinburg no NHL team, playing elsewhere
    Michael McLeod        C               NJD      False        KHL              Avangard Omsk no NHL team, playing elsewhere
       Matt Murray        G               SEA      False        NaN                        NaN  no 2026-27 games, no NHL team
      Patrik Laine        L               MTL      False        NaN                        NaN  no 2026-27 games, no NHL team
    Maxime Comtois        L               CAR      False        KHL             Spartak Moskva

## Takeaways
- **The snapshot:** 32 active rosters (797 players, 22 to 29 per team) and 2,503 fresh profiles, downloaded November 6, 2026. **Opening night was September 29, 2026** (FLA at CAR, MTL at TOR, NYR at BOS, VAN at EDM, CHI at VGK).
- **The NHL profile's team is the evidence.** It records who holds a player, signed or not, including traded rights. Examples:
  - Max Plante is DET while he's at Minnesota-Duluth.
  - Buchelnikov's rights are now with STL.
  - Command's rights went to WPG in the trade.
- **How teams were decided:**
  - **Team today:** the profile's team for 1,228 players and the active roster for 787. The parent club of an AHL/ECHL club for 56 (AHL contracts).
  - **Opening night:** 8 in-season moves with games were undone, plus the 5 players in the logged trade, and 2 overrides.
  - **Nobody claims him:** the CBA check applies only to draftees who never played in the NHL. It gave 37 their draft team and says 91 have rights that likely expired. Everyone else nobody claims (289) is not with an NHL organization.
- **Affiliates from the data:**
  - All 32 AHL clubs have a clear parent, one per NHL team.
  - Only 6 of 27 ECHL clubs do (most serve several parents or have too few NHL-signed players), so ECHL-contract players there have no team.
- **Opening-night counts:**
  - Active rosters run 22 to 28 per team. With players off the active roster and the two disputes, the NHL level is 22 to 31 (TOR 31).
  - **Every one of the 822 NHL-level players has a projection.**
- **The trade log checks out:** all five players in the Hellebuyck trade show their new team today. On opening night Hellebuyck is a Jet, and Mercer, Daws and Bjugstad are Devils. Bjugstad is off New Jersey's active roster; Command is a Devils prospect in Sweden's U20 league.
- **Six other in-season moves to confirm for the trade log** (from game logs):
  - Hallander PIT to EDM (Oct 10)
  - Struble MTL to NSH (Oct 10)
  - Hunt MIN to NSH (Oct 25)
  - St. Ivany WPG to WSH (Oct 28)
  - Shane Wright SEA to NSH (Nov 3)
  - Matthew Wood NSH to SEA (Nov 4)

  The baseline already starts each of them on his first team, so adding them to the log only matters for the simulator's in-season trades.
- **Detroit on opening night:**
  - **NHL level (28):** 26 on the active roster plus Larkin ("suspended / holdout") and Edvinsson ("unsigned RFA").
  - **AHL (17):** 16 on NHL contracts plus Kienan Draper (AHL contract).
  - **Rights held (28):** junior, college and Europe. Pradel and Bicker have no 2026-27 games yet, and Tnias Mathurin is held by the CBA check (profile blank).
  - Kane (CHI), Perron (DAL) and Talbot (CBJ) are with other teams, not out of the NHL. van Riemsdyk has no 2026-27 team.
- **Redmond Savage:** BUF, "AHL/ECHL (AHL contract)". Detroit's 2021 pick has no NHL team on his profile and plays for Rochester (Buffalo's AHL club), so the rule "AHL/ECHL club's NHL parent" decided it.
- **279 projected NHL players changed teams since 2025-26.** For example: Kyrou STL to WSH, Dorofeyev VGK to NYR, Brady Tkachuk OTT to FLA, Marchenko CBJ to TOR, Knies TOR to CBJ.
- **Prospects:** of notebook 19's 1,167, 959 are still with their draft team on opening night. 74 are with another team (traded rights or a new organization), and 134 have no team (91 rights likely expired, 43 not with an NHL organization).
- **The CBA check is rough (rules from memory):**
  - **29 players the profile keeps but the CBA says expired.** Mostly Russians and Swedes drafted 2020 to 2022 and playing in Europe, so European rights probably last longer than four years (for example, while he's under a European contract). The profile wins.
  - **37 players the profile drops but the CBA keeps.** Mostly NCAA players drafted 2021 to 2023, given their draft team and flagged "rights held, profile blank (CBA)". Check these before they matter.
- **289 projected players are not with an NHL organization** (246 NHL players, 43 prospects):
  - Playing in Europe with no NHL team: Milano and Skinner in Switzerland; Sprong, McLeod and Comtois in the KHL.
  - Inactive with no 2026-27 games: Matt Murray, Laine, Kopitar, van Riemsdyk.
  - **Ryan to confirm the list.**
- **New to our data (8)**, projected with notebook 19's prior (skaters) or notebook 21's (goalies), after both checks reproduced those notebooks exactly:
  - Pinchuk NSH 0.064 WAR/60 and Formenton EDM 0.052 lead.
  - Goalies Postava (DET) 0.044 and Poirier (WSH) 0.040.
- **Limits:**
  - "NHL roster" means today's active roster, so call-ups and injuries since opening night aren't undone; only trades and moves with games are.
  - The API can't tell injured, suspended and unsigned players apart, so overrides set those statuses.
  - Signed players who aren't on a roster and weren't projected before can't be found without contract data.
- **Files** (all gitignored):
  - `rosters_2026-27.csv` (2,503 players; 2,123 with an NHL team on opening night)
  - `affiliates_2026-27.csv`
  - `new_players_2026-27.csv`
  - `trades_2026-27.csv` and `roster_overrides.csv`, hand-kept and never overwritten
- **Next:** rerun notebook 20 so the forecast table uses opening-night teams and statuses, then minutes projections and the season simulator.